# SequentialSelector

Greedy selection with logistic regression. Forward: start empty and, at each
step, add the one feature that gives the best cross-validated score.
Backward: start with all features and remove one at a time.

- **Cost:** one cross-validation per candidate feature per step. Forward
  selection of k from p features costs about k × p × cv model fits.
- **Strength:** features are judged by the classifier's held-out score.
- **Weakness:** far too slow for thousands of features without a prefilter,
  and greedy: a feature chosen early is never reconsidered.
- **Classification only.** Wraps scikit-learn's `SequentialFeatureSelector`.

In [1]:
import time

import numpy as np
from sklearn.datasets import make_classification

from featsel.selectors import SequentialSelector

# Only 20 features: this method is slow. The first 4 are informative.
X, y = make_classification(n_samples=200, n_features=20, n_informative=4,
                           n_redundant=0, shuffle=False, random_state=0)
truth = set(range(4))

## Forward and backward

In [2]:
for direction in ('forward', 'backward'):
    start = time.perf_counter()
    chosen = set(SequentialSelector(n_features=4, direction=direction)
                 .fit(X, y).get_support(indices=True))
    print(f'{direction:<8} {time.perf_counter() - start:5.2f} s   '
          f'chosen {sorted(int(i) for i in chosen)}, {len(chosen & truth)} of 4 informative')

forward   1.86 s   chosen [1, 2, 11, 12], 2 of 4 informative


backward  6.46 s   chosen [0, 1, 2, 17], 3 of 4 informative


## Where it fails

Cost grows with the number of candidate features. Doubling the width roughly
doubles the time, and SCAN-B has over 9,000 features.

In [3]:
rng = np.random.default_rng(0)
for n_noise in (0, 20, 60):
    X_wide = np.column_stack([X, rng.normal(size=(len(y), n_noise))])
    start = time.perf_counter()
    SequentialSelector(n_features=4).fit(X_wide, y)
    print(f'{X_wide.shape[1]:3d} features: {time.perf_counter() - start:5.2f} s')

 20 features:  1.89 s


 40 features:  3.88 s


 80 features:  7.93 s
